# KAT-VAD v2 — P4 finish: **VideoMAE V2-S on full T2** + **D6 shuffle control** (no training)

E2(d) picked **`vit_s_k710_dl_from_giant`** (`v2_E2d/e2d_readout.md`, rule N7: A3 transfer Δ B 0.1508 vs S 0.1325,
gap 0.0183 < 0.02 → cheaper encoder). This notebook does the two P4 steps that follow the pick, as fixed in
`core/docs/v2/PREREG_ADDENDUM.md` **§15 (Amendment 6, G1–G4)** before any of their numbers exist:

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps | — |
| 1 | preflight: §15 in the tree, E2(d) pick == S, splits, census, caches | no |
| 2 | **V2-S on every T2 source** (train + T2-val + test; K's geometry, same cache dir — the 300 K sources are skipped) | **yes** |
| 3 | **D6**: re-stream the CAP videos of `dota_cap_dev` only, rebuild, pixel-gate (L4–L5), encode V2-S with every window's 16 frames **permuted** (seed 2024) → `DoTA_CAP_s1_squash_shuf2024` | **yes** |
| 4 | D6 read-out: in-domain probe, ordered vs shuffled (`e2d_shuffle`) | no |
| 5 | record | — |

Step 2 and step 3 are independent; both resume. **D6 is printed, never decided on** (it cannot move the pick).
Bake A2/A3 + pilot batch 2 are in `p6_pilot_batch2.ipynb` (needs step 2 only).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # data/, cache/ (VideoMAE + CLIP caches)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # kat-vad/ (code), outputs/ (v2 read-outs)
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V2_D6_SHUFFLE_SEED'), 'this upload predates Amendment 6 -- re-upload branch v2'

ENC_S    = constants.VIDEOMAE_ENCODER_S
SEED     = constants.V2_D6_SHUFFLE_SEED                     # 2024 (N11)
DATASET  = constants.DADA_ORIGIN_DATASET                    # 'DADA2000_orig'
STRIDE   = constants.FRAME_STRIDE                           # 8: K's geometry
T2       = constants.DATA_ROOT / DATASET
CENSUS   = T2 / 'counts' / 'resolved_census.json'
ORIG     = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
VIDEO    = constants.VIDEO_CACHE_DIR
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
RAW      = constants.DATA_ROOT / 'MMAU' / 'raw'
GROUPS   = ['1-10', '11', '12-42', '43', '44-62']           # CAP-DATA tar groups (dota_cap_videomae.ipynb)
BATCH_GB = constants.MMAU_STREAM_BATCH_GB
REPORTS  = constants.OUTPUT_ROOT / 'REPORTS'
E2D      = REPORTS / 'v2_E2d'
CAP_REP  = REPORTS / 'dota_cap'                             # alignment + the ordered extract reports
D6_REP   = REPORTS / 'v2_E2d_shuffle'
LOCAL    = Path('/content/p4s')                             # VM-local NVMe, never Drive
for p in (D6_REP, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name in ('T2', 'CENSUS', 'ORIG', 'DOTA_S1', 'RAW', 'E2D', 'CAP_REP', 'D6_REP'):
    print(f'{name:8s}', globals()[name])

In [ ]:
%%bash
apt-get -qq install -y p7zip-full
pip install -q "transformers==4.56.*" av einops faiss-cpu xlrd
python -c "import transformers, av, einops, faiss, xlrd; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 2 and 3 need one'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

ENV = dict(os.environ)


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')

## 1. Preflight — the registered pick, splits and caches

In [ ]:
from core.data.v2_splits import load_split
from core.tools import build_v2_inputs, dota_cap, e2d_probe, e2d_shuffle, extract_video_features

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 15. Amendment 6' in addendum, 'this upload predates addendum §15'
pick = json.loads((E2D / e2d_probe.READOUT_JSON).read_text())['decision']['pick']
assert pick == ENC_S, f'E2(d) picked {pick}, this notebook is written for {ENC_S}'
cap_dev = load_split(constants.V2_SPLIT_DOTA_CAP_DEV)                    # sha1-checked
assert len(cap_dev) == 569, len(cap_dev)

sources = build_v2_inputs.t2_all_sources(T2)                             # train + T2-val + test (what `fit` bakes)
census = json.loads(CENSUS.read_text())
missing = [v for v in sources if v not in census]
assert not missing, f'{len(missing)} T2 sources have no census entry: {missing[:5]}'
S_DIR = e2d_probe.t2_video_dir(VIDEO, ENC_S)
T2_MANIFEST = extract_video_features.build_manifest(ENC_S, STRIDE, constants.VIDEOMAE_WEIGHTS_SHA256[ENC_S])
e2d_probe.check_manifest(S_DIR, T2_MANIFEST)                              # the K-source cache step 2 extends (C2)

alignment_sha = e2d_probe.dota_cap_alignment_sha256(constants.V2_SPLITS_DIR)
assert dota_cap.file_sha256(CAP_REP / dota_cap.ALIGNMENT_JSON) == alignment_sha, 'alignment != the frozen one'
ORDERED_DIR = dota_cap.video_out_dir(ENC_S)
SHUF_DIR = dota_cap.video_out_dir(ENC_S, SEED)
e2d_probe.check_manifest(ORDERED_DIR, dota_cap.video_manifest(ENC_S, alignment_sha))
assert all((ORDERED_DIR / f'{v}.npy').exists() for v in cap_dev), 'ordered DoTA-CAP V2-S cache incomplete'
done = sum((S_DIR / f'{v}.npy').exists() for v in sources)
print(f'pick {pick} | T2 sources {len(sources)} ({done} already encoded) | dota_cap_dev {len(cap_dev)} | '
      f'frame order {extract_video_features.shuffled_frame_order(SEED)}')

## 2. V2-S on **every** T2 source — GPU, resumable (G2)

Same pipeline and geometry as K (`p1_kill_switch.ipynb` §3, `p4_e2d.ipynb` §2): 16 frames every 3rd at the assumed
30 fps (1.5 s), causal, squash 224², stride 8, into the **same** cache dir (its manifest is checked, C2). A shard whose
ids are all encoded is skipped, so the 300 K sources cost nothing. **Re-run this cell after a disconnect.**

In [ ]:
SHARD, BATCH = 60, 32
root, images = constants.DADA_ORIGIN_ROOT_DIRNAME, constants.DADA_ORIGIN_IMAGES_SUBDIR
assert (ORIG / 'DADA2000.zip').is_file(), f'no DADA2000.zip under {ORIG}'
todo = [v for v in sources if not (S_DIR / f'{v}.npy').exists()]
print(f'{len(todo)} of {len(sources)} T2 sources to encode')


def type_video(vid):
    t, v = vid.split('_')                      # t{type:02d}_v{video:03d}
    return int(t[1:]), int(v[1:])


started = time.monotonic()
for k in range(0, len(todo), SHARD):
    shard, tag = todo[k:k + SHARD], f'{k // SHARD:03d}'
    if all((S_DIR / f'{v}.npy').exists() for v in shard):
        continue
    frames, farm = LOCAL / 'frames', LOCAL / 'flat'
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    farm.mkdir(parents=True)
    patterns = [f'{root}/{t}/{v:03d}/{images}/*' for t, v in map(type_video, shard)]
    print(f'--- shard {tag}: {len(shard)} sources, {shutil.disk_usage("/content").free / 2**30:.1f} GiB free')
    run(['7z', 'x', ORIG / 'DADA2000.zip', f'-o{frames}', '-y', '-bso0', '-bsp0', *patterns])
    for vid in shard:                          # C26: link each source's images dir under its unique id
        t, v = type_video(vid)
        src = frames / root / str(t) / f'{v:03d}' / images
        n = len(list(src.glob('*')))
        if n != census[vid]:                   # C10: a folder is not evidence of data
            raise RuntimeError(f'{vid}: {n} images extracted, census says {census[vid]}')
        (farm / vid).symlink_to(src.resolve(), target_is_directory=True)
    shard_ids = LOCAL / f'ids_{tag}.txt'
    shard_ids.write_text('\n'.join(shard) + '\n')
    run([sys.executable, '-m', 'core.tools.extract_video_features',
         '--frames-dir', farm, '--ids-file', shard_ids, '--dataset', DATASET,
         '--encoder', ENC_S, '--stride', STRIDE, '--batch-size', BATCH,
         '--device', 'auto', '--output-dir', S_DIR])
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    print(f'    encoded {sum((S_DIR / f"{v}.npy").exists() for v in sources)}/{len(sources)} | '
          f'{(time.monotonic() - started) / 60:.1f} min')

assert all((S_DIR / f'{v}.npy').exists() for v in sources), 'full-T2 V2-S cache incomplete -- rerun this cell'
e2d_probe.check_manifest(S_DIR, T2_MANIFEST)
# G2 row check: one motion row per CLIP step of every source (what `fit` will pair them with)
import numpy as np  # noqa: E402
clip_dir = constants.CLIP_CACHE_DIR / DATASET
bad = [v for v in sources
       if len(np.load(S_DIR / f'{v}.npy', mmap_mode='r')) != len(np.load(clip_dir / f'{v}.npy', mmap_mode='r'))]
assert not bad, f'{len(bad)} sources: V2-S rows != CLIP rows: {bad[:5]}'
print(f'V2-S on all {len(sources)} T2 sources: complete, manifest == K geometry, rows == CLIP rows')

## 3. **D6** — shuffled V2-S on `dota_cap_dev` (GPU, G3)

`dota_cap extract --shuffle-seed 2024 --dota-ids-file dota_cap_dev`: only the CAP videos of the 569 DoTA-CAP-dev clips
are streamed; each DoTA clip is rebuilt and **pixel-gated again** (L4–L5, same thresholds), then encoded by V2-S with
the same fixed permutation of the 16 frame slots in every window. It writes `DoTA_CAP_s1_squash_shuf2024` and its own
manifest (`frame_order`, `shuffle_seed`); it can never write into, or be read as, the ordered cache.

In [ ]:
import pandas as pd

ann = pd.read_excel(RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
type_of = {f'{v:06d}': int(t) for v, t in zip(ann['video'].astype(int), ann['type'].astype(int))}
align = json.loads((CAP_REP / dota_cap.ALIGNMENT_JSON).read_text())['clips']
need_cap = {align[v]['cap_id'] for v in cap_dev}
assert all(align[v]['reason'] == 'ok' for v in cap_dev)
dota_ids_file = LOCAL / 'dota_cap_dev.txt'
dota_ids_file.write_text(''.join(f'{v}\n' for v in cap_dev))
free_gb = shutil.disk_usage('/content').free / 1e9
assert free_gb >= 3 * BATCH_GB, f'{free_gb:.0f} GB free on /content, need {3 * BATCH_GB:.0f} -- lower BATCH_GB'

for g in GROUPS:
    lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
    ids = sorted(c for c in need_cap if lo <= type_of[c] <= hi)
    print(f'group {g}: {len(ids)} CAP videos needed')
    if not ids:
        continue
    ids_file = LOCAL / f'cap_ids_{g}.txt'
    ids_file.write_text(''.join(f'{i}\n' for i in ids))
    parts = sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
    assert parts, f'no parts for group {g}'
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.dota_cap', 'extract',
         '--alignment', CAP_REP / dota_cap.ALIGNMENT_JSON, '--parts', *parts,
         '--work-dir', LOCAL / 'work' / g, '--dota-clip-dir', DOTA_S1,
         '--report', D6_REP / f'extract_shuf_{g}.json', '--cap-ids-file', ids_file,
         '--dota-ids-file', dota_ids_file, '--shuffle-seed', SEED,
         '--encoders', ENC_S, '--batch-size', '32', '--max-batch-gb', str(BATCH_GB),
         '--device', 'auto'], log=LOCAL / 'd6_extract.log')
    print(f'group {g}: {(time.time() - t0) / 60:.0f} min')

report = {}
for p in sorted(D6_REP.glob('extract_shuf_*.json')):
    report |= json.loads(p.read_text())
bad = {v: report.get(v, {}).get('reason', 'not_run') for v in cap_dev if report.get(v, {}).get('reason') != 'ok'}
assert not bad, f'{len(bad)} DoTA-CAP-dev clips failed the re-run gate (they passed it at extraction): {bad}'
assert all((SHUF_DIR / f'{v}.npy').exists() for v in cap_dev), 'shuffled cache incomplete -- rerun this cell'
e2d_probe.check_manifest(SHUF_DIR, dota_cap.video_manifest(ENC_S, alignment_sha, SEED))
print(f'D6 cache: {len(cap_dev)} clips, pixel gate re-passed on all, min clip cos '
      f'{min(report[v]["clip_min_cos"] for v in cap_dev):.4f}')

## 4. D6 read-out — ordered vs shuffled, in-domain probe (G3; printed only)

In [ ]:
stage = LOCAL / 'stage'
s1_local = stage / 'DoTA_s1_ncc'
dirs = e2d_shuffle.shuffle_dirs(stage / 'video', ENC_S, SEED)
jobs = [(DOTA_S1, s1_local), (ORDERED_DIR, dirs[e2d_shuffle.ORDERED]), (SHUF_DIR, dirs[e2d_shuffle.SHUFFLED])]
t0 = time.time()
for src, dst in jobs:
    dst.mkdir(parents=True, exist_ok=True)
    for v in cap_dev:
        if not (dst / f'{v}.npy').exists():
            shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
    manifest = src / constants.VIDEO_MANIFEST_FILENAME
    if manifest.exists():
        shutil.copy2(manifest, dst / manifest.name)
print(f'staged in {time.time() - t0:.0f}s')

run([sys.executable, '-m', 'core.tools.e2d_shuffle',
     '--dota-s1-dir', s1_local, '--metadata', DOTA / 'metadata_val.json',
     '--split-file', DOTA / 'val_split.txt', '--video-root', stage / 'video',
     '--encoder', ENC_S, '--shuffle-seed', SEED, '--out-dir', D6_REP], log=D6_REP / 'shuffle_run.log')
print((D6_REP / e2d_shuffle.READOUT_MD).read_text())

## 5. Record — then **stop**

Paste `v2_E2d_shuffle/shuffle_readout.md` back to Claude. Step 2's cache is what `p6_pilot_batch2.ipynb` bakes A2/A3
from; that notebook can start as soon as step 2 is complete (it does not need step 3/4).